# Kinship Library
## Notebook 02: NLP, Text Quality and Semantic Foundations

### Goal

Prepare text representations for both the bootcamp baseline and the later semantic recommender.

### Models in this notebook

TF IDF gives us an interpretable lexical baseline.

Sentence Transformers give us semantic embeddings that can later match a reader's natural language description with book content.

### Important

We do not diagnose readers.

A later app may invite a person to describe what they are exploring and retrieve books whose themes are semantically related to that text.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"
MODELS = PROJECT_ROOT / "models"

MODELS.mkdir(parents=True, exist_ok=True)

books = pd.read_csv(PROCESSED_DATA / "books_clean.csv")

print("Books loaded:", len(books))
books.head()

## 1. Text completeness

Descriptions are ideal for semantic search, but Open Library Search results may not provide a description for every book.

For the first baseline, we create a text field from verified metadata that we do have.

Title plus subjects plus collection query gives the model more context than title alone.

Later, verified descriptions can be added through enrichment without changing the structure of the project.

In [ ]:
books["semantic_text"] = (
    books["title"].fillna("") + ". "
    + books["subjects"].fillna("") + ". "
    + books["search_query"].fillna("")
)

books["semantic_text_length"] = (
    books["semantic_text"]
    .str.split()
    .str.len()
)

books[
    ["title", "semantic_text", "semantic_text_length"]
].head()

## 2. Simple text cleaning for TF IDF

TF IDF benefits from basic cleaning.

We keep a separate semantic text column because transformer models usually work better with natural text than aggressively cleaned text.

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-zA-ZÀ-ÿ\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text

books["clean_text"] = books["semantic_text"].apply(clean_text)

books[
    ["semantic_text", "clean_text"]
].head()

## 3. TF IDF baseline

### Purpose

Represent books according to weighted words and two word combinations.

### Expected result

Each book becomes a sparse numeric vector.

In [ ]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2
)

tfidf_matrix = tfidf.fit_transform(
    books["clean_text"]
)

print("TF IDF shape:", tfidf_matrix.shape)

## 4. Inspect important vocabulary

In [ ]:
feature_names = np.array(
    tfidf.get_feature_names_out()
)

mean_weights = np.asarray(
    tfidf_matrix.mean(axis=0)
).flatten()

top_indices = mean_weights.argsort()[::-1][:30]

top_terms = pd.DataFrame({
    "term": feature_names[top_indices],
    "mean_tfidf": mean_weights[top_indices]
})

top_terms

## 5. Reader path prompts

These are not diagnoses and they are not labels attached to the books.

They are example natural language queries that we can use to test semantic retrieval later.

In [ ]:
READER_PATHS = {
    "fear_of_the_future": (
        "I feel uncertain and afraid about the future and I want books "
        "that can help me explore change, uncertainty, responsibility, and hope."
    ),
    "ecological_concern": (
        "I feel overwhelmed by environmental destruction and I want to understand "
        "how humans can live with more care and responsibility toward the living world."
    ),
    "disconnection_from_nature": (
        "I feel disconnected from nature and I want to rebuild a meaningful "
        "relationship with land, animals, plants, and the more than human world."
    ),
    "search_for_meaning": (
        "I am looking for meaning and a deeper understanding of life, "
        "consciousness, purpose, and spiritual practice."
    ),
    "embodied_way_of_living": (
        "I want practical and philosophical guidance for living with more awareness "
        "through Yoga, meditation, breath, body, and daily practice."
    )
}

pd.DataFrame(
    READER_PATHS.items(),
    columns=["reader_path", "example_query"]
)

## 6. TF IDF query test

This gives us a baseline before transformers.

A lexical system works best when the reader uses words that also occur in the book metadata.

In [ ]:
def recommend_tfidf(user_text, number_of_books=5):
    query_vector = tfidf.transform(
        [clean_text(user_text)]
    )

    scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    best_indices = scores.argsort()[::-1][:number_of_books]

    results = books.iloc[best_indices][
        [
            "title",
            "author",
            "collection_theme",
            "subjects"
        ]
    ].copy()

    results["similarity_score"] = scores[best_indices]

    return results.reset_index(drop=True)

recommend_tfidf(
    READER_PATHS["disconnection_from_nature"]
)

## 7. Semantic embeddings with Sentence Transformers

### Why not plain RoBERTa?

RoBERTa is a powerful transformer language model, but a recommender needs comparable vectors for queries and books.

Sentence Transformer models are trained or structured specifically for sentence and document embeddings and semantic similarity.

This section is optional on the first run because Colab must download the model.

### Install once in Colab

Run the next cell if `sentence_transformers` is not already installed.

In [ ]:
# Uncomment in Colab if needed
# !pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

semantic_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

book_embeddings = semantic_model.encode(
    books["semantic_text"].tolist(),
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Semantic embedding shape:", book_embeddings.shape)

## 8. Semantic query test

Now the reader's wording does not need to match the exact vocabulary in the catalogue.

This is the first prototype of the future experience based recommender.

In [ ]:
def recommend_semantic(user_text, number_of_books=5):

    query_embedding = semantic_model.encode(
        [user_text],
        normalize_embeddings=True
    )

    scores = cosine_similarity(
        query_embedding,
        book_embeddings
    ).flatten()

    best_indices = scores.argsort()[::-1][:number_of_books]

    results = books.iloc[best_indices][
        [
            "title",
            "author",
            "collection_theme",
            "subjects"
        ]
    ].copy()

    results["semantic_score"] = scores[best_indices]

    return results.reset_index(drop=True)

recommend_semantic(
    READER_PATHS["disconnection_from_nature"]
)

## 9. Compare lexical and semantic retrieval

We should not assume the transformer is automatically better.

We compare the actual recommendations and later evaluate them manually with a small relevance table.

In [ ]:
test_query = READER_PATHS["search_for_meaning"]

print("TF IDF recommendations")
display(recommend_tfidf(test_query))

print("Semantic recommendations")
display(recommend_semantic(test_query))

## 10. Save features for the next notebook

In [ ]:
books.to_csv(
    PROCESSED_DATA / "books_nlp_ready.csv",
    index=False
)

np.save(
    MODELS / "book_embeddings.npy",
    book_embeddings
)

print("Saved NLP ready catalogue and semantic embeddings")